# Review detections against truth

A metric tells you how much is wrong. Looking tells you *what kind* of wrong —
a box in the right place with the wrong class, a box half the size it should be,
or nothing at all. Those three have different fixes and the same F1.

In [ ]:
import orinth

MODEL_ID = next(
    model.id
    for model in orinth.models.list()
    if model.available and model.task_type in {"object_detection", "segmentation"}
)
DATASET_ID = next(
    dataset.id
    for dataset in orinth.datasets.list()
    if dataset.task_type in {"object_detection", "segmentation"} and dataset.splits.get("valid")
)
predictor = orinth.models.predictor(MODEL_ID)
params = orinth.models.parameters()
frame = orinth.datasets.load(DATASET_ID, "valid", limit=24)
print(MODEL_ID, "on", DATASET_ID, "|", frame.height, "items")

## Score each image by how wrong it is

Class-set agreement, per image: what the annotator wrote versus what the model
said. Cheap, and it sorts the pile — which is all this needs to do before a
human looks.

In [ ]:
from pathlib import Path

import polars as pl

reviewed = []
for row in frame.iter_rows(named=True):
    truth = {annotation["label"] for annotation in (row["annotations"] or [])}
    detections = predictor.predict(Path(row["path"]), params)
    predicted = {detection.class_name for detection in detections}
    reviewed.append(
        {
            "path": row["path"],
            "item": Path(row["path"]).name,
            "truth": ", ".join(sorted(truth)) or "none",
            "predicted": ", ".join(sorted(predicted)) or "none",
            "missed": len(truth - predicted),
            "spurious": len(predicted - truth),
            "boxes_true": len(row["annotations"] or []),
            "boxes_pred": len(detections),
        }
    )

review = pl.DataFrame(reviewed).with_columns(
    (pl.col("missed") + pl.col("spurious")).alias("wrongness")
)
print(review.select(["missed", "spurious"]).sum())
review.sort("wrongness", descending=True).select(
    ["item", "truth", "predicted", "boxes_true", "boxes_pred"]
).head(10)

## Look at the worst four

Truth in one colour, prediction in another, on the same image. Everything about
the failure that a table cannot carry — position, size, whether the model found
the right thing and named it wrong — is visible in two seconds here.

In [ ]:
%matplotlib inline
import matplotlib.patches as patches
import matplotlib.pyplot as plt
from PIL import Image

worst = review.sort("wrongness", descending=True).head(4)
figure, axes = plt.subplots(1, worst.height, figsize=(4 * worst.height, 4.4))
axes = axes if worst.height > 1 else [axes]

for axis, row in zip(axes, worst.iter_rows(named=True)):
    with Image.open(row["path"]) as handle:
        axis.imshow(handle.convert("RGB"))
    source = frame.filter(pl.col("path") == row["path"]).row(0, named=True)

    for annotation in source["annotations"] or []:
        polygon = annotation.get("polygon")
        box = annotation.get("bbox")
        if polygon:
            axis.add_patch(patches.Polygon(polygon, closed=True, fill=False, linewidth=1.6, edgecolor="lime"))
        elif box:
            axis.add_patch(
                patches.Rectangle((box["x"], box["y"]), box["width"], box["height"],
                                  fill=False, linewidth=1.6, edgecolor="lime")
            )

    for detection in predictor.predict(Path(row["path"]), params):
        rect = detection.bbox
        axis.add_patch(
            patches.Rectangle((rect.x, rect.y), rect.width, rect.height,
                              fill=False, linewidth=1.6, edgecolor="orangered", linestyle="--")
        )

    axis.set_title(f"{row['item'][:18]}\ntruth {row['truth']} / pred {row['predicted']}", fontsize=8)
    axis.axis("off")

figure.tight_layout()
plt.show()
print("green = annotated · dashed red = predicted")

## Size, where detectors usually fail

If the misses cluster at the small end, the fix is resolution or anchor scale,
not more epochs. This compares the size of what was found against the size of
what was there.

In [ ]:
sizes = []
for row in frame.iter_rows(named=True):
    for annotation in row["annotations"] or []:
        box = annotation.get("bbox")
        polygon = annotation.get("polygon")
        if box:
            sizes.append({"kind": "annotated", "size": (box["width"] * box["height"]) ** 0.5})
        elif polygon:
            xs = [x for x, _ in polygon]
            ys = [y for _, y in polygon]
            sizes.append({"kind": "annotated", "size": ((max(xs) - min(xs)) * (max(ys) - min(ys))) ** 0.5})
    for detection in predictor.predict(Path(row["path"]), params):
        sizes.append({"kind": "predicted", "size": (detection.bbox.width * detection.bbox.height) ** 0.5})

pl.DataFrame(sizes).group_by("kind").agg(
    pl.len().alias("regions"),
    pl.col("size").median().round(1).alias("median_px"),
    pl.col("size").min().round(1).alias("smallest"),
)